# 00 — Extracción y exploración — LockBit Panel DB

Parsea `paneldb_dump.sql` (volcado MySQL del panel de administración de LockBit),  
guarda cada tabla como parquet y produce una exploración inicial del dataset.

**A diferencia de ContiLeaks/BlackBasta**, este dataset no son chats entre miembros  
del grupo — es la base de datos operacional del panel: víctimas, operadores, builds  
de malware y negociaciones de rescate con las víctimas.

Produce en `data/processed/`:
- `users.parquet`
- `clients.parquet`
- `chats.parquet`
- `builds.parquet`
- `btc_addresses.parquet`
- `invites.parquet`

## 0. Setup

In [ ]:
# "sys" nos da acceso a información del sistema, como la lista de carpetas donde
# Python busca módulos. Lo necesitamos para decirle dónde está nuestro código propio.
import sys

# "Path" de pathlib es la forma moderna de trabajar con rutas de archivos.
# Hace que construir rutas sea más claro y portable entre sistemas operativos.
from pathlib import Path

# Añadimos la carpeta 'src' al comienzo de la lista de rutas de búsqueda de Python.
# Así Python podrá encontrar e importar nuestro módulo 'loaders' que está en esa carpeta.
# str() lo convierte en texto porque sys.path espera cadenas, no objetos Path.
# .resolve() convierte la ruta relativa 'src' en la ruta absoluta completa.
sys.path.insert(0, str(Path('src').resolve()))

# "pandas" es la librería principal para análisis de datos en Python.
# Con ella trabajamos con tablas (DataFrames), similar a hojas de Excel.
import pandas as pd

# "matplotlib.pyplot" es la librería estándar de Python para hacer gráficas.
# La abreviamos como "plt" por convención en toda la comunidad Python.
import matplotlib.pyplot as plt

# "matplotlib.ticker" nos permite personalizar los ejes de las gráficas,
# por ejemplo, formatear los números como "1,000" en lugar de "1000".
import matplotlib.ticker as mticker

# "numpy" es la librería de cálculo numérico de Python (arrays, matemáticas).
# La abreviamos como "np" por convención.
import numpy as np

# Importamos nuestra función personalizada 'load_lockbit' desde el módulo 'loaders'
# que está en la carpeta 'src'. Esta función sabe leer el volcado SQL de LockBit.
from loaders import load_lockbit

# Definimos las rutas importantes del proyecto.
# SQL_ZIP: dónde está el archivo ZIP que contiene el volcado de la base de datos de LockBit.
SQL_ZIP      = Path('/home/drjekyll/Documentos/umbrella/data_bruto/Ransomware/Lockbit_paneldb_dump 2025.zip')

# PROCESSED: carpeta donde guardaremos los datos ya procesados en formato parquet.
# ".." significa "subir un nivel" en la jerarquía de carpetas desde donde está el notebook.
PROCESSED    = Path('data/processed')

# Creamos la carpeta PROCESSED si no existe.
# parents=True → crea también las carpetas padre si son necesarias
# exist_ok=True → no da error si la carpeta ya existe
PROCESSED.mkdir(parents=True, exist_ok=True)

# Verificamos que el archivo ZIP existe antes de continuar.
# assert lanza un error con el mensaje si la condición es falsa,
# lo cual es útil para detectar problemas de configuración de forma clara.
assert SQL_ZIP.exists(), f'No se encuentra {SQL_ZIP}'

# Imprimimos el nombre del archivo y su tamaño en megabytes para confirmar que lo encontramos.
# SQL_ZIP.stat().st_size devuelve el tamaño en bytes; dividimos entre 1024² para convertir a MB.
# :.1f formatea el número con 1 decimal.
print(f'Archivo: {SQL_ZIP.name}  ({SQL_ZIP.stat().st_size/1024**2:.1f} MB)')

## 1. Carga y extracción

In [ ]:
# Avisamos al usuario de que el proceso va a comenzar. Como el archivo SQL es grande,
# puede tardar varios segundos. El mensaje ayuda a saber que el programa no se colgó.
print('Parseando SQL dump (dos pasadas: schema + datos)...')

# Llamamos a nuestra función 'load_lockbit' pasándole la ruta del archivo ZIP.
# Esta función hace todo el trabajo pesado: leer el SQL, separar las tablas,
# convertir los tipos de datos y devolver un diccionario de DataFrames.
# 'tables' es un diccionario: cada clave es el nombre de una tabla ('clients', 'chats', etc.)
# y cada valor es un DataFrame de pandas con los datos de esa tabla.
tables = load_lockbit(SQL_ZIP)

# Mostramos un resumen de todas las tablas que se extrajeron correctamente.
print('\n=== TABLAS EXTRAÍDAS ===')

# Recorremos el diccionario de tablas en orden alfabético (sorted()).
# Para cada tabla imprimimos:
#   - su nombre (alineado en 20 caracteres con :20s)
#   - cuántas filas tiene (len(df))
#   - cuántas columnas tiene (len(df.columns))
# La coma en :6, formatea el número con separador de miles: 59975 → 59,975
for name, df in sorted(tables.items()):
    print(f'  {name:20s}: {len(df):6,} filas x {len(df.columns):2d} cols')

In [ ]:
# Guardar como parquet (excluimos pkeys por tamaño y sensibilidad)
# Parquet es un formato de archivo columnar muy eficiente: ocupa mucho menos espacio
# que un CSV y se carga mucho más rápido en futuros notebooks.
# Excluimos 'pkeys' (30.000 filas de claves criptográficas) porque no la analizaremos
# directamente y contiene datos sensibles.
SAVE_TABLES = ['users', 'clients', 'chats', 'builds', 'btc_addresses', 'invites']

# Guardamos cada tabla en su propio archivo .parquet dentro de la carpeta PROCESSED
for name in SAVE_TABLES:
    # Construimos la ruta de salida: PROCESSED / 'nombre_tabla.parquet'
    out = PROCESSED / f'{name}.parquet'

    # .to_parquet() convierte el DataFrame a formato parquet y lo escribe en disco.
    # index=False evita guardar el índice numérico (0, 1, 2, ...) como columna extra,
    # ya que no nos aporta información útil.
    tables[name].to_parquet(out, index=False)

    # Mostramos el nombre del archivo creado y su tamaño en kilobytes para verificar.
    # out.stat().st_size devuelve el tamaño en bytes; dividimos entre 1024 para KB.
    print(f'  {out.name}  ({out.stat().st_size/1024:.0f} KB)')

print('\nParquets guardados.')

## 2. Operadores / Afiliados

In [ ]:
# Extraemos la tabla de usuarios del diccionario de tablas.
# 'users' contiene los operadores y afiliados del panel de LockBit:
# las personas que usaban el ransomware para atacar víctimas.
users = tables['users']

# Imprimimos estadísticas básicas sobre los usuarios del panel.
print('=== OPERADORES Y AFILIADOS ===')

# len(users) devuelve el número total de filas (registros) en la tabla
print(f'Total registrados : {len(users)}')

# Contamos cuántos usuarios tienen el campo is_admin igual a 1.
# La comparación (users.is_admin == 1) devuelve una serie de True/False,
# y .sum() cuenta los True (Python trata True=1 y False=0 en sumas).
print(f'Administradores   : {(users.is_admin == 1).sum()}')

# El 'level' indica el rango del usuario dentro de la organización.
# Nivel 4 es el nivel más alto (administrador del panel).
print(f'Nivel 4 (admin)   : {(users.level == 4).sum()}')

# El 'tag' categoriza a los afiliados según su estado de verificación.
# 'newbie' = recién registrado, aún no verificado por los administradores.
print(f'Con tag "newbie"  : {(users.tag == "newbie").sum()}')

# 'verified' = afiliado confiable que ya ha demostrado ser legítimo.
print(f'Con tag "verified": {(users.tag == "verified").sum()}')
print()

# Mostramos una vista previa de los primeros 15 usuarios con las columnas más relevantes.
# display() es una función especial de Jupyter que muestra DataFrames de forma visual,
# con mejor formato que print(). Solo funciona dentro de notebooks.
# Seleccionamos solo las columnas que más nos interesan con la lista entre corchetes.
print('Primeros 15 usuarios (login, nivel, tag, último acceso):')
display(users[['id','login','is_admin','level','tag','last_online_dt']].head(15))

## 3. Víctimas (clients)

In [ ]:
# Extraemos la tabla de clientes (víctimas) del diccionario de tablas.
# En el panel de LockBit, las víctimas se llaman 'clients' — una terminología
# perturbadora que muestra la mentalidad empresarial de los criminales.
clients = tables['clients']

# Imprimimos estadísticas básicas sobre las víctimas.
print('=== VÍCTIMAS ===')

# Número total de víctimas en la base de datos
print(f'Total víctimas         : {len(clients)}')

# Contamos cuántas víctimas tienen al menos un mensaje de chat.
# clients.id.isin(tables["chats"].clientid) devuelve True para cada víctima
# cuyo 'id' aparece en la columna 'clientid' de la tabla de chats.
# .sum() cuenta cuántos True hay.
print(f'Con actividad de chat  : {clients.id.isin(tables["chats"].clientid).sum()}')

# Cuántas víctimas pagaron la comisión del afiliado (es decir, pagaron el rescate)
print(f'Pagaron comisión       : {(clients.paid_commission == 1).sum()}')

# Cuántas víctimas completaron el proceso de descifrado de sus archivos
# (decrypt_done > 0 significa que el descifrado fue exitoso)
print(f'Descifrado completado  : {(clients.decrypt_done > 0).sum()}')

# Cuántas víctimas fueron 'baneadas' del panel (casos cancelados o problemáticos)
print(f'Baneadas               : {(clients.banned == 1).sum()}')
print()

# Analizamos qué operadores tienen más víctimas asignadas.
# 'advid' (advertiser ID) identifica qué operador/afiliado maneja cada víctima.
# groupby('advid') → agrupa las filas por operador
# .size() → cuenta cuántas víctimas tiene cada operador
# .sort_values(ascending=False) → ordena de mayor a menor
op_victims = clients.groupby('advid').size().sort_values(ascending=False)

# Mostramos los 10 operadores más "productivos" (los que tenían más víctimas)
print('Top 10 operadores por víctimas asignadas (advid):')
# .to_string() convierte la serie a texto para mostrarlo sin el dtype al final
print(op_victims.head(10).to_string())

In [ ]:
# Timeline de compromisos: cuántas víctimas nuevas aparecían cada semana.

# Primero eliminamos las filas donde 'date_first' es NaT (fecha no disponible).
# dropna(subset=['date_first']) elimina solo las filas sin fecha, conservando el resto.
# 'date_first' es la fecha en que la víctima accedió por primera vez al panel de LockBit.
clients_valid = clients.dropna(subset=['date_first'])

# Agrupamos las víctimas por semana y contamos cuántas nuevas hay cada semana.
# set_index('date_first') → usamos la fecha como índice (eje temporal)
# .resample('W') → reagrupamos los datos en intervalos de una semana ('W' = week)
# .size() → contamos cuántas filas (víctimas) hay en cada semana
weekly = clients_valid.set_index('date_first').resample('W').size()

# Creamos una figura con un único gráfico de barras.
# figsize=(13, 4) define el tamaño en pulgadas: 13 de ancho, 4 de alto.
fig, ax = plt.subplots(figsize=(13, 4))

# Dibujamos las barras: el eje X es la semana, el eje Y es el número de víctimas.
# width=5 hace las barras más anchas para que se vean mejor en una escala semanal.
ax.bar(weekly.index, weekly.values, width=5)

# Añadimos etiquetas descriptivas para que el gráfico sea autoexplicativo
ax.set_title('Nuevas víctimas por semana — LockBit Panel')
ax.set_xlabel('Semana')
ax.set_ylabel('Víctimas')

# Ajustamos automáticamente los márgenes para que nada se corte
plt.tight_layout()

# Mostramos el gráfico en el notebook
plt.show()

# Imprimimos la semana con más víctimas nuevas (el "pico" de actividad del ransomware)
# weekly.idxmax() devuelve el índice (fecha) donde el valor es máximo
# weekly.max() devuelve el valor máximo
print(f'Pico: {weekly.idxmax().date()}  ({weekly.max()} víctimas/semana)')

# Mostramos el rango temporal completo del dataset
# .min() y .max() devuelven la fecha más antigua y más reciente respectivamente
# .date() formatea la fecha como YYYY-MM-DD, sin la hora
print(f'Rango temporal: {clients_valid.date_first.min().date()} → {clients_valid.date_first.max().date()}')

## 4. Builds de malware

In [ ]:
# Extraemos la tabla 'builds': cada fila es un ejecutable de ransomware compilado.
# Los operadores generaban builds personalizados para cada víctima o campaña,
# con configuraciones específicas (servidor C2, nota de rescate, etc.).
builds = tables['builds']

# Estadísticas básicas sobre los builds de malware
print('=== BUILDS ===')

# Total de ejecutables compilados durante el período analizado
print(f'Total builds           : {len(builds)}')

# Cuántos operadores distintos compilaron builds (un mismo operador puede tener varios)
# .nunique() cuenta los valores únicos en una columna (nunca cuenta el mismo dos veces)
print(f'Operadores distintos   : {builds.userid.nunique()}')
print()

# Distribución por 'type': el número indica qué variante de LockBit se usó.
# Por ejemplo: 25=LockBit 2.0, 30=LockBit 3.0, 40=LockBit 4.0, etc.
# .value_counts() cuenta cuántas veces aparece cada valor y ordena de mayor a menor.
print('Distribución por tipo:')
print(builds.type.value_counts().to_string())
print()

# Distribución por 'revenue': el rango de ingresos estimados de la víctima.
# Los operadores seleccionaban el tamaño del rescate según el tamaño de la empresa.
# Valores como '5kk' = 5 millones, '10k' = 10 mil, etc.
print('Distribución por revenue range:')
if 'revenue' in builds.columns:
    # .head(10) muestra solo los 10 valores más frecuentes para no saturar la pantalla
    print(builds.revenue.value_counts().head(10).to_string())

In [ ]:
# Evolución temporal de builds: cuántos ejecutables se compilaban cada semana.

# Eliminamos las filas sin fecha de compilación para no distorsionar el análisis
builds_valid = builds.dropna(subset=['date'])

# Agrupamos por semana y contamos builds por semana, igual que hicimos con las víctimas
builds_weekly = builds_valid.set_index('date').resample('W').size()

# Diccionario que asigna un color específico a cada variante de LockBit.
# Esto permite distinguir visualmente las versiones en el gráfico.
# Los colores son códigos hexadecimales (formato web estándar, ej: '#e74c3c' = rojo).
type_colors = {25: '#e74c3c', 30: '#3498db', 40: '#2ecc71', 46: '#f39c12', 50: '#9b59b6'}

# Creamos una figura con dos gráficos lado a lado (1 fila, 2 columnas).
# figsize=(14, 5) → 14 pulgadas de ancho, 5 de alto
# 'axes' es una lista con los dos objetos de gráfico: axes[0] (izquierda) y axes[1] (derecha)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico izquierdo: distribución de builds por variante de LockBit ---

# Contamos cuántos builds hay de cada tipo y los ordenamos por número de tipo
type_counts = builds.type.value_counts().sort_index()

# Asignamos el color correspondiente a cada tipo; si no está en el diccionario, usamos gris
colors = [type_colors.get(t, '#95a5a6') for t in type_counts.index]

# Dibujamos las barras. Convertimos los tipos a texto (str) para que aparezcan como categorías.
axes[0].bar([str(t) for t in type_counts.index], type_counts.values, color=colors)
axes[0].set_title('Builds por variante de LockBit')
axes[0].set_xlabel('Tipo')
axes[0].set_ylabel('Nº builds')

# --- Gráfico derecho: evolución temporal de compilaciones por semana ---

# Dibujamos un gráfico de barras temporal: el eje X es la fecha, Y es el nº de builds
axes[1].bar(builds_weekly.index, builds_weekly.values, width=5)
axes[1].set_title('Builds por semana')
axes[1].set_xlabel('Semana')
axes[1].set_ylabel('Nº builds')

# Ajustamos los espacios para que los dos gráficos no se solapen
plt.tight_layout()

# Mostramos ambas gráficas en el notebook
plt.show()

## 5. Chats de negociación (vista previa)

In [ ]:
# Extraemos la tabla de chats: los mensajes de negociación de rescate.
# Cada fila es un mensaje enviado entre un operador de LockBit y una víctima.
# Esta tabla es el corazón del análisis cualitativo: aquí está la comunicación real.
chats = tables['chats']

# Estadísticas generales del canal de comunicación criminal-víctima
print('=== CHATS DE NEGOCIACIÓN ===')

# Total de mensajes intercambiados en todas las negociaciones
# :, formatea el número con separador de miles para mejor legibilidad
print(f'Total mensajes       : {len(chats):,}')

# Cuántas víctimas distintas participaron en conversaciones
# .nunique() cuenta los valores únicos, ignorando repeticiones
print(f'Víctimas distintas   : {chats.clientid.nunique()}')

# Cuántos operadores distintos gestionaron negociaciones
print(f'Operadores distintos : {chats.advid.nunique()}')

# Mensajes enviados por operadores (flag=1 significa que el remitente es el operador)
print(f'Mensajes operador    : {(chats.flag==1).sum():,}')

# Mensajes enviados por víctimas (flag=0 significa que el remitente es la víctima)
print(f'Mensajes víctima     : {(chats.flag==0).sum():,}')

# Mensajes que son archivos adjuntos (pruebas de desencriptado, capturas, documentos)
print(f'Con adjuntos         : {(chats.is_file==1).sum()}')

# Rango de fechas que cubre el dataset
print(f'Rango                : {chats.created_at.min().date()} → {chats.created_at.max().date()}')
print()

# Mostramos un ejemplo real de negociación para ilustrar el proceso.
# Elegimos la primera víctima que aparece en los datos (iloc[0] = primera fila)
print('Ejemplo de negociación (victim 24):')
sample = chats[chats.clientid == chats.clientid.iloc[0]].sort_values('created_at')

# Recorremos los primeros 6 mensajes de esa negociación
for _, row in sample.head(6).iterrows():
    # Etiquetamos claramente quién envió cada mensaje
    sender = 'OPERADOR' if row.flag == 1 else 'VÍCTIMA '

    # Mostramos solo los primeros 100 caracteres del mensaje para no saturar la pantalla.
    # replace('\n', ' ') elimina los saltos de línea para que quepa en una línea.
    # Si no hay contenido (es un archivo adjunto), mostramos '[adjunto]'.
    msg = str(row.content)[:100].replace('\n', ' ') if row.content else '[adjunto]'

    # Mostramos la hora (%H:%M), quién lo envió y el contenido abreviado
    print(f"  [{row.created_at.strftime('%H:%M')}] {sender}: {msg}")

## 6. Invites (estructura de reclutamiento)

In [ ]:
# Extraemos la tabla de 'invites': los códigos de invitación para reclutar afiliados.
# LockBit funciona como un modelo RaaS (Ransomware as a Service): los criminales
# pagan una cuota de entrada para acceder al panel y al malware.
invites = tables['invites']

# Estadísticas del sistema de reclutamiento
print('=== INVITES / AFILIACIÓN ===')

# Total de invitaciones generadas (la mayoría nunca fueron usadas)
print(f'Total invites    : {len(invites)}')

# status=50 significa que la invitación fue aceptada y el afiliado se unió
print(f'Aceptadas (50)   : {(invites.status == 50).sum()}')

# status=0 significa invitación pendiente, esperando ser usada
print(f'Pendientes (0)   : {(invites.status == 0).sum()}')

# Cuántas invitaciones incluyen una wallet de Bitcoin como método de pago
# .notna() devuelve True para los valores que NO son nulos (hay wallet)
print(f'Con wallet BTC   : {invites.btc_wallet.notna().sum()}')

# Cuántas invitaciones incluyen una wallet de Monero (criptomoneda más anónima que BTC)
print(f'Con wallet XMR   : {invites.monero_wallet.notna().sum()}')

# Analizamos los montos pagados para unirse a LockBit como afiliado
# pd.to_numeric() convierte la columna a números; errors='coerce' convierte errores a NaN
# .dropna() elimina los NaN para hacer los cálculos solo con valores válidos
amounts = pd.to_numeric(invites.amount, errors='coerce').dropna()

if len(amounts) > 0:
    print(f'\nMontos de afiliación (BTC/XMR):')
    # Mostramos estadísticas descriptivas: mínimo, máximo y mediana
    # :.6f formatea los números con 6 decimales (necesario para criptomonedas)
    print(f'  Mín: {amounts.min():.6f}')
    print(f'  Máx: {amounts.max():.6f}')
    # La mediana es el valor central (la mitad paga menos, la otra mitad paga más)
    # Es más robusta que la media cuando hay valores extremos
    print(f'  Mediana: {amounts.median():.6f}')

## 7. Resumen final del dataset

In [ ]:
# Resumen ejecutivo del dataset completo en forma de tabla ASCII.
# Este bloque construye una "caja" de texto para presentar los hallazgos clave
# de forma visual y fácil de leer, como si fuera un informe resumido.

# Imprimimos el borde superior de la caja
print('╔══════════════════════════════════════════════╗')
print('║          RESUMEN — LOCKBIT PANEL DB          ║')
print('╠══════════════════════════════════════════════╣')

# Total de operadores/afiliados registrados en el panel
# >6, formatea el número alineado a la derecha en 6 espacios, con separador de miles
print(f'║  Operadores registrados : {len(users):>6,}           ║')

# Total de víctimas en el sistema
print(f'║  Víctimas comprometidas : {len(clients):>6,}           ║')

# Cuántas víctimas pagaron el rescate (paid_commission=1)
print(f'║  Víctimas que pagaron   : {(clients.paid_commission==1).sum():>6,}           ║')

# Tasa de conversión: porcentaje de víctimas que terminaron pagando.
# Dividimos los que pagaron entre el total y multiplicamos por 100 para obtener el porcentaje.
# >5.1f alinea el número en 5 espacios con 1 decimal.
print(f'║  Tasa de conversión     :  {(clients.paid_commission==1).sum()/len(clients)*100:>5.1f}%           ║')

# Total de ejecutables de ransomware compilados
print(f'║  Builds de malware      : {len(builds):>6,}           ║')

# Total de mensajes intercambiados en negociaciones
print(f'║  Mensajes de negociación: {len(chats):>6,}           ║')

# Total de invitaciones de reclutamiento generadas
print(f'║  Invites de afiliación  : {len(invites):>6,}           ║')

# Total de direcciones Bitcoin para recibir pagos (altísimo volumen)
print(f'║  Direcciones Bitcoin    : {len(tables["btc_addresses"]):>6,}           ║')

# Período temporal que cubre el dataset
print(f'║  Período cubierto       : dic 2024 – abr 2025   ║')

# Borde inferior de la caja
print('╚══════════════════════════════════════════════╝')